# 코랩 음식 사진 배경 교체 검증

운영과 같은 안전 모드로 실행합니다. 기본 탐지기는 학습한 음식 전용 YOLO11n `models/best.pt`입니다. YOLO11n 탐지 실패 시 중앙 사각형을 사용하지 않으며, `food_detection_failed` 보고서를 남기고 광고 이미지를 만들지 않습니다.

In [ ]:
from google.colab import drive, files
from pathlib import Path
drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/final_1_team/apps/api/food-image-cleanup-pipeline')
assert (PROJECT_ROOT / 'configs/pipeline.yaml').is_file(), f'프로젝트를 찾을 수 없습니다: {PROJECT_ROOT}'
%cd $PROJECT_ROOT

In [ ]:
import os, subprocess, sys
PACKAGE_DIR = Path('/content/food-image-cleanup-packages')
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '--target', str(PACKAGE_DIR), '--prefer-binary', '-r', 'requirements-colab.txt'], check=True)
RUNTIME_ENV = os.environ.copy()
RUNTIME_ENV['PYTHONPATH'] = str(PACKAGE_DIR) + os.pathsep + RUNTIME_ENV.get('PYTHONPATH', '')
print('파이프라인 의존성 설치 완료')

In [ ]:
# 배경 교체에 필요한 기본 모델만 다운로드합니다. 학습한 best.pt와 efficientnet_best.pt는 Drive의 프로젝트 폴더에 직접 있어야 합니다.
subprocess.run([sys.executable, '-m', 'scripts.download_models', '--models', 'yolo', 'sam2', 'big-lama', 'openclip', 'birefnet', 'sana'], check=True, env=RUNTIME_ENV)
DETECTOR_PROFILE = 'food_specialized'  # 비교가 필요하면 coco_yolo11n으로 변경합니다.
DETECTOR_WEIGHTS = PROJECT_ROOT / 'models/best.pt' if DETECTOR_PROFILE == 'food_specialized' else PROJECT_ROOT / 'models/yolo11n.pt'
ANGLE_WEIGHTS = PROJECT_ROOT / 'models/efficientnet_best.pt'
assert DETECTOR_WEIGHTS.is_file(), f'선택한 탐지기 가중치가 없습니다: {DETECTOR_WEIGHTS}'
assert ANGLE_WEIGHTS.is_file(), f'EfficientNet-B0 각도 분류 가중치가 없습니다: {ANGLE_WEIGHTS}'
print(f'탐지 프로필: {DETECTOR_PROFILE}, 가중치: {DETECTOR_WEIGHTS}')
print(f'각도 분류기: {ANGLE_WEIGHTS}')

In [ ]:
import json, shutil
from PIL import Image
from IPython.display import display
uploaded = files.upload()
assert len(uploaded) == 1, '음식 사진 한 장만 업로드하세요.'
source_path = Path(next(iter(uploaded)))
input_path = Path('data/input') / f'example{source_path.suffix.lower()}'
input_path.parent.mkdir(parents=True, exist_ok=True)
shutil.move(str(source_path), input_path)
metadata = {'business_type':'cafe', 'food_category':'dessert', 'foreground_position':'center_lower', 'light_direction':'left'}
metadata_path = Path('data/input/example_metadata.json')
metadata_path.write_text(json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8')
display(Image.open(input_path))

In [ ]:
# --diagnostic-center-fallback을 넣지 않습니다. 탐지 실패는 운영과 같이 안전하게 차단됩니다.
command = [sys.executable, '-m', 'scripts.run_background_replacement', '--input', str(input_path), '--metadata', str(metadata_path), '--enable-matting', '--enable-background-generator', '--detector-profile', DETECTOR_PROFILE]
result = subprocess.run(command, cwd=PROJECT_ROOT, text=True, capture_output=True, env=RUNTIME_ENV)
print(result.stdout)
if result.stderr: print(result.stderr)
print('종료 코드:', result.returncode)

In [ ]:
from IPython.display import Image as DisplayImage
report_path = Path('data/reports') / f'{input_path.stem}_background_replacement_report.json'
assert report_path.is_file(), f'보고서가 없습니다: {report_path}'
report = json.loads(report_path.read_text(encoding='utf-8'))
detector_stage = report.get('stages', {}).get('step_2_yolo_detection', {})
angle_stage = report.get('stages', {}).get('step_7_camera_angle_classification', {})
assert detector_stage.get('profile') == DETECTOR_PROFILE, f'탐지 프로필 불일치: {detector_stage}'
if DETECTOR_PROFILE == 'food_specialized':
    assert detector_stage.get('model', '').replace('\\', '/').endswith('models/best.pt'), detector_stage
assert angle_stage.get('model', '').replace('\\', '/').endswith('models/efficientnet_best.pt'), angle_stage
assert angle_stage.get('status') in {'completed', 'low_confidence'}, angle_stage
assert angle_stage.get('label') in {'top', '45'}, angle_stage
print(json.dumps({'status':report.get('status'), 'reason':report.get('reason'), 'detector':detector_stage, 'camera_angle':angle_stage, 'debug_artifacts':report.get('debug_artifacts'), 'validation':report.get('stages',{}).get('step_13_foreground_validation')}, ensure_ascii=False, indent=2))
for name, artifact_path in report.get('debug_artifacts', {}).items():
    artifact = Path(artifact_path)
    if artifact.is_file():
        print(name, artifact)
        display(DisplayImage(filename=str(artifact)))